# Análisis GEE: Tratamiento × SDO × Gap
**Análisis principal y exploratorio — muestra completa y subgrupo NFC alta**

Este notebook asume que el dataframe limpio ya fue generado por el notebook de limpieza.  
El análisis utiliza exclusivamente las observaciones del **primer bloque recibido** por cada participante (diseño efectivamente entre-sujeto).

Variables esperadas en el dataframe:
- `sujeto_id` — identificador único del participante
- `condicion` — str: 'DIST', 'SIN', 'CON'
- `gap` — float: brecha en ARS (0, 1000, 1200, 2000, 2400)
- `mantiene` — int: 1 = preserva jerarquía, 0 = revierte
- `sdo` — float: puntaje SDO (escala continua)
- `ndc_score` — float: puntaje NFC (escala continua)
- `orden_1` — int: 1 si este bloque fue el primero recibido por el participante

> **Ajustar** la variable `RUTA_DATOS` si el archivo tiene otro nombre o ubicación.

---
## 0. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.families import Binomial
from statsmodels.genmod.cov_struct import Exchangeable
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', 20)

# ── AJUSTAR RUTA ──────────────────────────────────────────────────────────────
RUTA_DATOS = 'datos_limpios.csv'   # CSV generado por el notebook de limpieza
# ─────────────────────────────────────────────────────────────────────────────

---
## 1. Carga y verificación del dataframe

In [ ]:
df_full = pd.read_csv(RUTA_DATOS)

print('Dimensiones dataframe completo:', df_full.shape)
print('\nColumnas disponibles:')
print(df_full.dtypes)
print('\nDistribución de condición:')
print(df_full['condicion'].value_counts())

In [ ]:
# ── Filtro entre-sujeto: primer bloque recibido únicamente ───────────────────
df = df_full[df_full['orden_1'] == 1].copy()

n_sujetos = df['sujeto_id'].nunique()
n_obs     = len(df)
print(f'Muestra analítica: {n_sujetos} sujetos, {n_obs} observaciones')
print('\nDistribución por condición (entre-sujeto):')
print(df.groupby('condicion')['sujeto_id'].nunique())

---
## 2. Estadística descriptiva de variables individuales

In [ ]:
# Una fila por sujeto para descriptivos de variables individuales
df_suj = df.drop_duplicates(subset='sujeto_id')

vars_desc = ['ndc_score', 'sdo']
# Agregar variables opcionales si existen
for v in ['ideologia', 'nse_subjetivo']:
    if v in df_suj.columns:
        vars_desc.append(v)

desc = df_suj[vars_desc].agg(['min', 'max', 'mean', 'std']).T
desc.columns = ['Min', 'Max', 'Media', 'DS']
print('Tabla X — Estadística descriptiva de variables individuales')
print(desc.round(2))

---
## 3. Existencia basal de ARR — Pruebas de Wilcoxon

### 3.1. Por condición (Tabla S4)

In [ ]:
def wilcoxon_vs_azar(serie, nombre='', alpha_ref=0.50):
    """
    Prueba de Wilcoxon de rangos con signo contra un valor de referencia.
    Devuelve estadístico W, p-valor bilateral, n.
    """
    dif = serie - alpha_ref
    dif_nonzero = dif[dif != 0]
    if len(dif_nonzero) == 0:
        return dict(variable=nombre, n=len(serie), M=serie.mean(),
                    DS=serie.std(), W=np.nan, p=np.nan)
    W, p = stats.wilcoxon(dif_nonzero, alternative='two-sided')
    return dict(variable=nombre, n=len(serie),
                M=serie.mean(), DS=serie.std(), W=W, p=p)

# Tasa de 'mantiene' por sujeto (media de los 6 ensayos del bloque)
tasa_suj = (df.groupby(['sujeto_id', 'condicion'])['mantiene']
              .mean()
              .reset_index()
              .rename(columns={'mantiene': 'tasa_mantiene'}))

rows = []
for cond in ['DIST', 'SIN', 'CON']:
    sub = tasa_suj[tasa_suj['condicion'] == cond]['tasa_mantiene']
    rows.append(wilcoxon_vs_azar(sub, nombre=cond))

tabla_s4 = pd.DataFrame(rows).set_index('variable')
print('Tabla S4 — Wilcoxon: Tasa de Mantiene vs. azar (0.50) por condición')
print(tabla_s4.round(3))

### 3.2. Por nivel de Gap (Tabla S5)

In [ ]:
# Tasa de 'mantiene' por sujeto × Gap
tasa_gap = (df.groupby(['sujeto_id', 'gap'])['mantiene']
              .mean()
              .reset_index()
              .rename(columns={'mantiene': 'tasa_mantiene'}))

rows = []
for gap_val in sorted(df['gap'].unique()):
    sub = tasa_gap[tasa_gap['gap'] == gap_val]['tasa_mantiene']
    rows.append(wilcoxon_vs_azar(sub, nombre=f'Gap={int(gap_val)}'))

tabla_s5 = pd.DataFrame(rows).set_index('variable')
print('Tabla S5 — Wilcoxon: Tasa de Mantiene vs. azar (0.50) por nivel de Gap')
print(tabla_s5.round(3))

---
## 4. Centrado de variables continuas

SDO centrado en M muestral.  
Gap centrado en M muestral y expresado en miles de ARS.

In [ ]:
# ── Medias muestrales (muestra analítica) ─────────────────────────────────────
M_SDO = df['sdo'].mean()
M_GAP = df['gap'].mean()         # en ARS
M_NDC = df['ndc_score'].mean()

print(f'M_SDO  = {M_SDO:.4f}')
print(f'M_Gap  = {M_GAP:.1f} ARS')
print(f'M_NDC  = {M_NDC:.4f}')

# ── Centrado ──────────────────────────────────────────────────────────────────
df['sdo_c']   = df['sdo']       - M_SDO
df['gap_k']   = (df['gap']      - M_GAP) / 1000   # miles de ARS, centrado
df['ndc_c']   = df['ndc_score'] - M_NDC

# ── Dummies de tratamiento (referencia = DIST) ────────────────────────────────
df['T_CON'] = (df['condicion'] == 'CON').astype(int)
df['T_SIN'] = (df['condicion'] == 'SIN').astype(int)

print('\nPrimeras filas con variables transformadas:')
print(df[['sujeto_id', 'condicion', 'gap', 'mantiene',
          'sdo_c', 'gap_k', 'ndc_c', 'T_CON', 'T_SIN']].head(8))

---
## 5. Modelo GEE — Muestra completa (Tabla 1)

**Especificación:** GEE binomial, link logit, correlación exchangeable,  
estimador sandwich Huber-White, agrupado por sujeto.  
Interacción triple Tratamiento × SDO × Gap con todos los términos constitutivos.

In [ ]:
# Ordenar por sujeto (requerimiento de GEE en statsmodels)
df_sorted = df.sort_values('sujeto_id').reset_index(drop=True)

formula_full = (
    'mantiene ~ T_CON + T_SIN + sdo_c + gap_k '
    '+ T_CON:sdo_c + T_SIN:sdo_c '
    '+ T_CON:gap_k + T_SIN:gap_k '
    '+ sdo_c:gap_k '
    '+ T_CON:sdo_c:gap_k + T_SIN:sdo_c:gap_k'
)

modelo_full = GEE.from_formula(
    formula       = formula_full,
    groups        = 'sujeto_id',
    data          = df_sorted,
    family        = Binomial(),
    cov_struct    = Exchangeable()
)

res_full = modelo_full.fit(cov_type='robust')
print(res_full.summary())

In [ ]:
def tabla_gee(resultado, titulo='Modelo GEE'):
    """
    Formatea la tabla de resultados GEE con β, SE, OR, IC 95%, p.
    Marca en negrita (texto) los efectos con p < .10.
    """
    params = resultado.params
    bse    = resultado.bse
    pvals  = resultado.pvalues
    ci     = resultado.conf_int()

    tbl = pd.DataFrame({
        'β':          params.round(3),
        'SE':         bse.round(3),
        'OR':         np.exp(params).round(3),
        'OR_IC_low':  np.exp(ci[0]).round(3),
        'OR_IC_high': np.exp(ci[1]).round(3),
        'p':          pvals.round(3),
    })
    tbl['sig'] = tbl['p'].apply(
        lambda p: '***' if p < .001 else
                  '**'  if p < .01  else
                  '*'   if p < .05  else
                  '†'   if p < .10  else ''
    )
    print(f'\n{titulo}')
    print('=' * 75)
    print(tbl.to_string())
    print(f'\nN observaciones : {int(resultado.nobs)}')
    print(f'N sujetos (grupos): {resultado.model.num_group}')
    return tbl

tbl_full = tabla_gee(
    res_full,
    titulo='Tabla 1 — GEE Tratamiento × SDO × Gap — Muestra completa'
)

---
## 6. Modelo GEE — Subgrupo NFC alta (Tabla 2)

**Criterio:** NDC ≥ percentil 75 de la muestra analítica.  
Análisis exploratorio — misma especificación que el modelo principal.

In [ ]:
# ── Definición del subgrupo ───────────────────────────────────────────────────
p75_ndc = df_sorted['ndc_score'].quantile(0.75)
print(f'Percentil 75 de NDC: {p75_ndc:.4f}')

df_nfc = df_sorted[df_sorted['ndc_score'] >= p75_ndc].copy()

n_suj_nfc = df_nfc['sujeto_id'].nunique()
n_obs_nfc = len(df_nfc)
print(f'Subgrupo NFC alta: {n_suj_nfc} sujetos, {n_obs_nfc} observaciones')
print('\nDistribución por condición en subgrupo:')
print(df_nfc.groupby('condicion')['sujeto_id'].nunique())

In [ ]:
# Misma fórmula que el modelo principal
modelo_nfc = GEE.from_formula(
    formula       = formula_full,
    groups        = 'sujeto_id',
    data          = df_nfc,
    family        = Binomial(),
    cov_struct    = Exchangeable()
)

res_nfc = modelo_nfc.fit(cov_type='robust')

tbl_nfc = tabla_gee(
    res_nfc,
    titulo='Tabla 2 — GEE Tratamiento × SDO × Gap — Subgrupo NFC alta'
)

---
## 7. Diagnósticos y verificaciones

### 7.1. Distribución de SDO — Justificación del resultado nulo

In [ ]:
df_suj_an = df_sorted.drop_duplicates(subset='sujeto_id')

sdo_vals = df_suj_an['sdo']
pct_bajo_2 = (sdo_vals < 2.0).mean() * 100

print('Distribución de SDO en muestra analítica:')
print(f'  M = {sdo_vals.mean():.3f}, DS = {sdo_vals.std():.3f}')
print(f'  Min = {sdo_vals.min():.2f}, Max = {sdo_vals.max():.2f}')
print(f'  % con SDO < 2.0 : {pct_bajo_2:.1f}%')
print(f'  Asimetría       : {sdo_vals.skew():.3f}')
print(f'  Curtosis        : {sdo_vals.kurtosis():.3f}')

print('\nPercentiles SDO:')
print(sdo_vals.describe(percentiles=[.10, .25, .50, .75, .90]).round(3))

### 7.2. Verificación de estructura de correlación exchangeable

In [ ]:
# Correlación intra-sujeto estimada por el modelo
print('Correlación exchangeable estimada (muestra completa):')
print(f'  ρ = {res_full.cov_struct.dep_params:.4f}')

print('\nCorrelación exchangeable estimada (subgrupo NFC):')
print(f'  ρ = {res_nfc.cov_struct.dep_params:.4f}')

### 7.3. Balance de celdas en el subgrupo NFC

In [ ]:
balance = (df_nfc.drop_duplicates('sujeto_id')
                 .groupby('condicion')['sujeto_id']
                 .count()
                 .rename('n_sujetos'))
print('Balance por condición en subgrupo NFC alta:')
print(balance)
print('\nNota: CON con n muy pequeño — coeficientes T_CON inestables.')

---
## 8. Exportar tablas a CSV

In [ ]:
tbl_full.to_csv('tabla1_GEE_muestra_completa.csv')
tbl_nfc.to_csv('tabla2_GEE_subgrupo_NFC.csv')
tabla_s4.to_csv('tabla_S4_wilcoxon_condicion.csv')
tabla_s5.to_csv('tabla_S5_wilcoxon_gap.csv')

print('Archivos exportados:')
print('  tabla1_GEE_muestra_completa.csv')
print('  tabla2_GEE_subgrupo_NFC.csv')
print('  tabla_S4_wilcoxon_condicion.csv')
print('  tabla_S5_wilcoxon_gap.csv')

---
## NOTAS DE INTERPRETACIÓN

```
RESULTADO NULO PRINCIPAL
El único predictor robusto en la muestra completa es el efecto principal de Gap.
Ninguna interacción con SDO alcanzó significancia (todas p > .15).

CAUSA PROBABLE
SDO tiene distribución severamente sesgada hacia el polo bajo
(M=2.12, DS=0.75; 44% < 2.0). La varianza disponible para detectar
interacciones es insuficiente. El resultado nulo no es evidencia
contra la hipótesis teórica — es evidencia contra la sensibilidad
del diseño para testearla con esta muestra.

SUBGRUPO NFC ALTA
T_SIN × SDO: β=1.333, p=.065 — efecto marginal, no sobrevive
corrección por comparaciones múltiples. Análisis estrictamente
exploratorio. n_CON=7 invalida cualquier inferencia sobre T_CON.
```